In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# SỬA ĐƯỜNG DẪN PROJECT_ROOT NẾU DỰ ÁN KHÔNG Ở VỊ TRÍ NÀY
PROJECT_ROOT = Path(r"D:\THAO_KLTN\stock_news_anomaly_project")
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "hose" / "raw_stock_prices_HOSE_13_2021_2026_complete_source_16198.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "hose_preprocessing"

START = pd.Timestamp("2021-10-19")
END = pd.Timestamp("2026-09-25")
TICKERS = {"ANV","DBC","DGW","DHG","FRT","IMP","MSN","MWG","PNJ","SAB","SBT","VHC","VNM"}
OHLC = ["open","high","low","close"]
NUMERIC = OHLC + ["volume"]

assert RAW_FILE.is_file(), (
    f"Không tìm thấy file RAW:\n{RAW_FILE}\n"
    "Hãy kiểm tra PROJECT_ROOT hoặc RAW_FILE ở cell đầu."
)
assert RAW_FILE.resolve() != OUTPUT_DIR.resolve(), "Không được ghi đè thư mục dữ liệu nguồn"
print("RAW:", RAW_FILE)
print("OUTPUT:", OUTPUT_DIR)

RAW: D:\THAO_KLTN\stock_news_anomaly_project\data\raw\hose\raw_stock_prices_HOSE_13_2021_2026_complete_source_16198.csv
OUTPUT: D:\THAO_KLTN\stock_news_anomaly_project\data\interim\hose_preprocessing


In [2]:
raw = pd.read_csv(
    RAW_FILE, sep=";", encoding="utf-8-sig",
    dtype={"ticker": "string", "date": "string"}, low_memory=False
)
raw.columns = raw.columns.str.strip()
required = {"ticker", "date", "market_record_status", "price_unit", "volume_unit", "source", *NUMERIC}
missing = required - set(raw.columns)
assert not missing, f"Thiếu cột: {sorted(missing)}"

df = raw.copy()
df["ticker"] = df["ticker"].str.strip().str.upper()
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="raise")
for col in NUMERIC:
    df[col] = pd.to_numeric(df[col], errors="raise")

assert len(df) == 16198, f"Số dòng RAW không đúng kỳ vọng: {len(df)}"
assert not df[["ticker","date"]].isna().any().any(), "Ticker/date bị thiếu"
assert not df[["ticker","date"]].duplicated().any(), "Trùng ticker-date"
assert set(df["ticker"]) == TICKERS, f"Tập mã khác kỳ vọng: {sorted(set(df['ticker']) ^ TICKERS)}"
assert df["date"].min() <= START and df["date"].max() >= END, "RAW không phủ đủ khoảng nghiên cứu"
assert df["price_unit"].eq("VND").all(), "Đơn vị giá chưa chuẩn VND"
assert df["volume_unit"].eq("shares_matched").all(), "Sai đơn vị volume"
assert df["source"].eq("HOSE_quote_report").all(), "Sai nguồn"
print("PASS RAW:", len(df), "dòng,", df["ticker"].nunique(), "mã")
display(df.head(3))

PASS RAW: 16198 dòng, 13 mã


,ticker,date,open,high,low,close,volume,price_unit,volume_unit,source,fetched_at_utc,price_verification_status,market_record_status,prior_close_reference_vnd,evidence_json
0,ANV,2021-10-19,32450.0,32450.0,32000.0,32300.0,468200,VND,shares_matched,HOSE_quote_report,2026-10-19T14:23:08.463950+00:00,NOT_PART_OF_492_AUDIT,MATCHED_TRADE,NaN,NaN
1,DBC,2021-10-19,56300.0,57100.0,56000.0,56100.0,1235300,VND,shares_matched,HOSE_quote_report,2026-10-19T14:23:08.463950+00:00,NOT_PART_OF_492_AUDIT,MATCHED_TRADE,NaN,NaN
2,DGW,2021-10-19,111500.0,115000.0,110000.0,114400.0,1164400,VND,shares_matched,HOSE_quote_report,2026-10-19T14:23:08.463950+00:00,NOT_PART_OF_492_AUDIT,MATCHED_TRADE,NaN,NaN


In [3]:
aligned = (
    df.loc[df["date"].between(START, END)]
    .sort_values(["ticker","date"]).reset_index(drop=True).copy()
)
dates = pd.DatetimeIndex(sorted(aligned["date"].unique()))
counts = aligned.groupby("ticker").size().reindex(sorted(TICKERS))
assert len(aligned) == 15990 and len(dates) == 1230, (
    f"Sai số dòng/ngày: {len(aligned)} dòng, {len(dates)} ngày"
)
assert counts.eq(1230).all(), f"Có mã thiếu dòng: {counts.to_dict()}"
assert aligned["date"].min() == START and aligned["date"].max() == END, "Sai ranh giới"
assert aligned.groupby("date")["ticker"].nunique().eq(13).all(), "Ngày thiếu ticker"
assert np.isfinite(aligned[NUMERIC].to_numpy(dtype=float)).all(), "OHLCV chứa NaN/Inf"
assert aligned["volume"].ge(0).all() and aligned["volume"].mod(1).eq(0).all(), "Volume lỗi"

matched = aligned["market_record_status"].eq("MATCHED_TRADE")
no_trade = aligned["market_record_status"].eq("NO_MATCHED_TRADE")
assert (matched | no_trade).all(), "Trạng thái giao dịch lạ"

valid_prices = aligned[OHLC].gt(0).all(axis=1)
valid_range = (
    aligned["high"].ge(aligned[OHLC].max(axis=1))
    & aligned["low"].le(aligned[OHLC].min(axis=1))
)
assert not (matched & (~valid_prices | ~valid_range | aligned["volume"].le(0))).any(), "OHLCV ở phiên khớp lệnh không hợp lệ"
assert not (no_trade & (aligned[NUMERIC].ne(0).any(axis=1) | aligned["ticker"].ne("IMP"))).any(), "No-trade record sai"
assert int(no_trade.sum()) == 13, f"Không phải 13 phiên IMP: {int(no_trade.sum())}"

print("PASS CUT:", len(aligned), "dòng,", len(dates), "ngày")
print("MATCHED_TRADE:", int(matched.sum()), "| NO_MATCHED_TRADE:", int(no_trade.sum()))
display(counts.rename("rows").to_frame())

PASS CUT: 15990 dòng, 1230 ngày
MATCHED_TRADE: 15977 | NO_MATCHED_TRADE: 13


,rows
ticker,
ANV,1230
DBC,1230
DGW,1230
DHG,1230
FRT,1230
IMP,1230
MSN,1230
MWG,1230
PNJ,1230


In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_aligned = OUTPUT_DIR / "01_hose_aligned_15990.csv"
out_imp = OUTPUT_DIR / "02_imp_no_matched_trade_13.csv"
assert out_aligned.resolve() != RAW_FILE.resolve(), "Không ghi đè RAW!"

aligned.to_csv(out_aligned, sep=";", index=False, encoding="utf-8-sig", date_format="%d/%m/%Y")
aligned.loc[no_trade].to_csv(out_imp, sep=";", index=False, encoding="utf-8-sig", date_format="%d/%m/%Y")
print("Đã lưu:", out_aligned.name, "và", out_imp.name)

Đã lưu: 01_hose_aligned_15990.csv và 02_imp_no_matched_trade_13.csv
